In [2]:
import pandas as pd

# Load CSV file
df = pd.read_csv("Hindi_Product Review_Augmented_Dataset.csv")

# Show first 5 reviews
print(df.head())


                                                text  polarity
0  ओह, मुझे यह फोन बहुत पसंद है। यह आश्चर्यजनक है...  positive
1  इस फोन से प्यार है, मेरे पास केवल एक समस्या थी...  positive
2  अरे, मैं आपको बताना चाहता हूं कि आपको लगता है ...  positive
3  बहुत बढ़िया फोन जब मैंने इसे ऑर्डर किया तो मैं...  positive
4  आसानी से सबसे अच्छा मोबाइल .. मैंने सभी हग एंड...  positive


In [3]:
print(df.head())



                                                text  polarity
0  ओह, मुझे यह फोन बहुत पसंद है। यह आश्चर्यजनक है...  positive
1  इस फोन से प्यार है, मेरे पास केवल एक समस्या थी...  positive
2  अरे, मैं आपको बताना चाहता हूं कि आपको लगता है ...  positive
3  बहुत बढ़िया फोन जब मैंने इसे ऑर्डर किया तो मैं...  positive
4  आसानी से सबसे अच्छा मोबाइल .. मैंने सभी हग एंड...  positive


In [4]:
corpus = df["text"].tolist()

print("Total Reviews:", len(corpus))
print("First Review:", corpus[0])


Total Reviews: 41682
First Review: ओह, मुझे यह फोन बहुत पसंद है। यह आश्चर्यजनक है। जैसे ही मुझे यह मिला मैंने पानी का परीक्षण किया। और यह काम करता है। मैंने इसे दो तरह से टेक्स्ट किया दोनों परीक्षणों पर स्पीकर थोड़ा मफल करता है मुझे लगता है कि जब तक यह सूख नहीं जाता। लेकिन ज्यादातर पीपीएल अपने फोन को पानी में डुबाने के लिए इधर-उधर नहीं जाते हैं, लेकिन अगर किसी कारण से आपका फोन पानी में खत्म हो जाता है तो चिंता की कोई बात नहीं है।


In [9]:
tokens = []

for sentence in corpus:
    words = str(sentence).split()
    tokens.extend(words)

print("Total Tokens:", len(tokens))
print("Sample Tokens:", tokens[:20])


Total Tokens: 1386662
Sample Tokens: ['ओह,', 'मुझे', 'यह', 'फोन', 'बहुत', 'पसंद', 'है।', 'यह', 'आश्चर्यजनक', 'है।', 'जैसे', 'ही', 'मुझे', 'यह', 'मिला', 'मैंने', 'पानी', 'का', 'परीक्षण', 'किया।']


In [10]:
from nltk.util import ngrams
from collections import Counter

unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))

unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)

print("Top 10 Unigrams:", unigram_counts.most_common(10))
print("Top 10 Bigrams:", bigram_counts.most_common(10))


Top 10 Unigrams: [(('के',), 37493), (('और',), 35057), (('है',), 33934), (('है।',), 30351), (('फोन',), 30246), (('में',), 27969), (('से',), 27402), (('यह',), 27153), (('नहीं',), 24246), (('एक',), 17650)]
Top 10 Bigrams: [(('के', 'लिए'), 11340), (('है', 'और'), 7994), (('के', 'साथ'), 5826), (('है', 'कि'), 5023), (('इस', 'फोन'), 4666), (('मेरे', 'पास'), 4104), (('यह', 'फोन'), 3949), (('करने', 'के'), 2971), (('फोन', 'को'), 2943), (('कर', 'रहा'), 2829)]


In [11]:
def bigram_probability(w1, w2):
    return bigram_counts[(w1, w2)] / unigram_counts[(w1,)]

print("P(फोन | यह) =", bigram_probability("यह", "फोन"))
print("P(लिए | के) =", bigram_probability("के", "लिए"))


P(फोन | यह) = 0.14543512687364196
P(लिए | के) = 0.3024564585389273


In [12]:
V = len(set(tokens))

def laplace_bigram(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[(w1,)] + V)

print("Smoothed P(शानदार | फोन) =", laplace_bigram("फोन", "शानदार"))
print("Smoothed P(खराब | फोन) =", laplace_bigram("फोन", "खराब"))


Smoothed P(शानदार | फोन) = 0.0008493552621419196
Smoothed P(खराब | फोन) = 0.0017450389931279438


In [13]:
import math

def sentence_probability(sentence):
    words = sentence.split()
    sentence_bigrams = list(ngrams(words, 2))

    prob = 1
    for w1, w2 in sentence_bigrams:
        prob *= laplace_bigram(w1, w2)

    return prob


def perplexity(sentence):
    words = sentence.split()
    N = len(words)

    P = sentence_probability(sentence)

    return math.pow(1/P, 1/N)


test_sentence = "यह फोन बहुत अच्छा है"

print("Test Sentence:", test_sentence)
print("Sentence Probability:", sentence_probability(test_sentence))
print("Perplexity:", perplexity(test_sentence))


Test Sentence: यह फोन बहुत अच्छा है
Sentence Probability: 4.6040755983633684e-07
Perplexity: 18.50851225249135
